# 📊 Caderno 3: Validação de Métricas e Engenharia de Dados (Interativo)

Este caderno permite validar a consistência matemática e analisar visualmente a volatilidade e tendências de diferentes ativos da B3 de forma interativa.

## 1. Configuração e Carga de Dados

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox
import warnings

warnings.filterwarnings('ignore')

# Carrega o Dataset Analítico
caminho_dados = '../data/processed/01_market_data_processed.csv'
df_processado = pd.read_csv(caminho_dados, sep=';', decimal=',')
df_processado['date'] = pd.to_datetime(df_processado['date'])

tickers_disponiveis = sorted(df_processado['ticker'].unique())
print(f"✅ Base de dados carregada! {len(tickers_disponiveis)} ativos encontrados.")

## 2. Dashboard de Validação Interativa

Escolha um ativo abaixo para verificar a consistência matemática e visualizar os indicadores técnicos.

In [ ]:
# Widgets de Controle
drop_ticker = widgets.Dropdown(options=tickers_disponiveis, value='PETR4.SA', description='Ativo:', style={'description_width': 'initial'})
check_mm21 = widgets.Checkbox(value=True, description='Mostrar MM21')
check_mm200 = widgets.Checkbox(value=False, description='Mostrar MM200')

def dashboard_interativo(ticker, show_mm21, show_mm200):
    df_at = df_processado[df_processado['ticker'] == ticker].sort_values('date').copy()
    
    # --- 1. Validação de Matemática ---
    preco_ini = df_at['close'].iloc[0]
    preco_fim = df_at['close'].iloc[-1]
    var_puro = (preco_fim / preco_ini) - 1
    ret_acum = df_at['retorno_acumulado'].iloc[-1]
    
    status = "✅ PERFEITO" if round(var_puro, 4) == round(ret_acum, 4) else "❌ ERRO DETECTADO"
    
    print(f"--- 🧪 CONSISTÊNCIA ({ticker}) ---")
    print(f"Status: {status}")
    print(f"Variação de Preço: {var_puro:.2%} | Retorno Acumulado: {ret_acum:.2%}")
    print("-" * 40)

    # --- 2. Gráfico de Preços e Médias ---
    fig_prep = go.Figure()
    fig_prep.add_trace(go.Scatter(x=df_at['date'], y=df_at['close'], name='Preço Fechamento', line=dict(color='white')))
    
    if show_mm21:
        fig_prep.add_trace(go.Scatter(x=df_at['date'], y=df_at['mm21'], name='MM21 (Curto Prazo)', line=dict(color='cyan', dash='dot')))
    if show_mm200:
        fig_prep.add_trace(go.Scatter(x=df_at['date'], y=df_at['mm200'], name='MM200 (Longo Prazo)', line=dict(color='yellow', width=1)))
        
    fig_prep.update_layout(title=f"Tendência de Preços: {ticker}", template='plotly_dark', height=400, xaxis_title="Data", yaxis_title="Preço (R$)")
    fig_prep.show()

    # --- 3. Gráfico de Volatilidade ---
    fig_vol = px.line(
        df_at, x='date', y='volatilidade_21d', 
        title=f"Risco e Volatilidade (21d): {ticker}",
        labels={'volatilidade_21d': 'Volatilidade Anualizada'},
        template='plotly_dark'
    )
    fig_vol.add_hline(y=0.60, line_dash="dash", line_color="red", annotation_text="Área de Pânico (>60%)")
    fig_vol.update_layout(height=400)
    fig_vol.show()

out = interactive_output(dashboard_interativo, {'ticker': drop_ticker, 'show_mm21': check_mm21, 'show_mm200': check_mm200})

display(VBox([HBox([drop_ticker, check_mm21, check_mm200]), out]))